# ResolveX — 00: Data Profiling (run LOCALLY)

Light-weight, fast (~1 minute on the full dataset). No blocking/training here, so
there's no benefit to running this on Kaggle — run it on your own machine.

Produces `experiments/reports/data_profile.md` and `experiments/metrics/data_profile.json`.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))


In [2]:
import json
import re

import pandas as pd

from resolvex.config import (
    METRICS_DIR, REPORTS_DIR,
    TEST_SOURCE1, TEST_SOURCE2, TEST_SOURCE3,
    TRAIN_GROUND_TRUTH, TRAIN_SOURCE1, TRAIN_SOURCE2, TRAIN_SOURCE3,
)
from resolvex.data_loader import load_ground_truth, load_source

NON_ASCII_RE = re.compile(r"[^\x00-\x7F]")
PUNCT_RE = re.compile(r"[^\w\s]")
DIGIT_RE = re.compile(r"\d")


In [3]:
def profile_source(df: pd.DataFrame) -> dict:
    name_col, addr_col = df["business_name"], df["business_address"]
    return {
        "row_count": len(df),
        "duplicate_entity_ids": int(df["entity_id"].duplicated().sum()),
        "duplicate_business_name_rows": int(name_col.duplicated().sum()),
        "duplicate_address_rows": int(addr_col.duplicated().sum()),
        "empty_name": int((name_col.str.len() == 0).sum()),
        "empty_address": int((addr_col.str.len() == 0).sum()),
        "empty_country": int((df["country"].str.len() == 0).sum()),
        "name_len_mean": float(name_col.str.len().mean()),
        "name_len_p95": float(name_col.str.len().quantile(0.95)),
        "address_len_mean": float(addr_col.str.len().mean()),
        "address_len_p95": float(addr_col.str.len().quantile(0.95)),
        "names_with_non_ascii": int(name_col.str.contains(NON_ASCII_RE, regex=True, na=False).sum()),
        "addresses_with_digits": int(addr_col.str.contains(DIGIT_RE, regex=True, na=False).sum()),
        "country_distribution": df["country"].value_counts().to_dict(),
    }


def profile_ground_truth(gt: pd.DataFrame) -> dict:
    match_counts = gt["matched_entity_ids"].apply(lambda s: 0 if not s else len(s.split(",")))
    n = len(gt)
    return {
        "n_source1_entities": n,
        "n_singleton": int((match_counts == 0).sum()),
        "singleton_frequency": float((match_counts == 0).mean()),
        "n_multi_match": int((match_counts > 1).sum()),
        "multi_match_frequency": float((match_counts > 1).mean()),
        "avg_match_count": float(match_counts.mean()),
        "max_match_count": int(match_counts.max()),
    }


In [4]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "train_source1": TRAIN_SOURCE1, "train_source2": TRAIN_SOURCE2, "train_source3": TRAIN_SOURCE3,
    "test_source1": TEST_SOURCE1, "test_source2": TEST_SOURCE2, "test_source3": TEST_SOURCE3,
}
all_stats = {}
for name, path in files.items():
    print(f"profiling {name}...")
    df = load_source(path)
    all_stats[name] = profile_source(df)
    del df

gt = load_ground_truth(TRAIN_GROUND_TRUTH)
all_stats["train_ground_truth"] = profile_ground_truth(gt)

with open(METRICS_DIR / "data_profile.json", "w", encoding="utf-8") as f:
    json.dump(all_stats, f, indent=2, ensure_ascii=False)

all_stats


profiling train_source1...
profiling train_source2...
profiling train_source3...
profiling test_source1...
profiling test_source2...
profiling test_source3...


{'train_source1': {'row_count': 2206821,
  'duplicate_entity_ids': 0,
  'duplicate_business_name_rows': 667592,
  'duplicate_address_rows': 76215,
  'empty_name': 0,
  'empty_address': 0,
  'empty_country': 0,
  'name_len_mean': 24.03440333402664,
  'name_len_p95': 37.0,
  'address_len_mean': 52.06621289175697,
  'address_len_p95': 103.0,
  'names_with_non_ascii': 0,
  'addresses_with_digits': 2129784,
  'country_distribution': {'US': 1323633, 'India': 883188}},
 'train_source2': {'row_count': 5034616,
  'duplicate_entity_ids': 0,
  'duplicate_business_name_rows': 632607,
  'duplicate_address_rows': 697354,
  'empty_name': 0,
  'empty_address': 168967,
  'empty_country': 0,
  'name_len_mean': 25.103574930044317,
  'name_len_p95': 40.0,
  'address_len_mean': 46.22590858965212,
  'address_len_p95': 96.0,
  'names_with_non_ascii': 764608,
  'addresses_with_digits': 4563675,
  'country_distribution': {'US': 3016817, 'India': 2017799}},
 'train_source3': {'row_count': 5285603,
  'duplicate_

In [5]:
lines = ["# ResolveX — Data Profile Report", ""]
for name in files:
    s = all_stats[name]
    lines.append(f"## {name}")
    for k, v in s.items():
        lines.append(f"- {k}: {v}")
    lines.append("")
lines.append("## train_ground_truth")
for k, v in all_stats["train_ground_truth"].items():
    lines.append(f"- {k}: {v}")

with open(REPORTS_DIR / "data_profile.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"Wrote {REPORTS_DIR / 'data_profile.md'}")


Wrote D:\Projects\ResolveX\experiments\reports\data_profile.md
